# Actividad 5 — Clasificación de abandono bancario
**Datos sintéticos, 800 registros.** Comparación de regresión logística y random forest, validación cruzada 5-fold, GridSearchCV y MLflow. No se usan datos reales.

## 1. Preparar el proyecto
Sube el archivo `Actividad5_repositorio.zip` a Colab cuando se solicite. El notebook descomprime el proyecto, instala dependencias y ejecuta el entrenamiento. La GPU no es necesaria para estos modelos scikit-learn.

## Cargar los archivos del proyecto en Colab
Sube `Actividad5_entrega.zip` (archivo de esta entrega). Así funciona antes de crear el repositorio remoto. Alternativamente, clona tu repositorio una vez publicado.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded=files.upload()
nombre=next((x for x in uploaded if x.endswith(".zip")),None)
assert nombre, "Sube Actividad5_entrega.zip"
with zipfile.ZipFile(nombre) as z: z.extractall("/content")
os.chdir("/content/Actividad5")
print("Carpeta actual:",os.getcwd())

In [ ]:
import pandas as pd
from fuentes.datos_prep import preparar_archivo
from pathlib import Path
base=Path('/content/Actividad5')
df=preparar_archivo(base/'datos/datos_ini/clientes_sinteticos_v1.csv',base/'datos/datos_limp/clientes_limpios_v1.csv')
print(df.shape,df['abandono'].value_counts().to_dict())
display(df.head())

## 3. Visualizaciones y control de calidad

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(10,4))
df['abandono'].value_counts().sort_index().plot.bar(ax=ax[0],title='Abandono (0=no, 1=sí)')
df.boxplot(column='meses_inactividad',by='abandono',ax=ax[1]);plt.suptitle('');plt.tight_layout();plt.show()

## 4. Entrenar, optimizar y registrar en MLflow
Ejecuta ambos modelos y registra parámetros, métricas y artefactos. El tracking queda en `/content/Actividad5/mlruns`.

In [ ]:
!python fuentes/train.py

## 5. Comparar resultados

In [ ]:
res=pd.read_csv('resultados/comparacion_modelos.csv')
display(res[['modelo','accuracy','precision','recall','f1','roc_auc','cv_f1_mean','cv_f1_std','best_params']])
from IPython.display import Image,display
display(Image(filename='resultados/roc_comparacion.png'))

## 6. Verificar registros de MLflow (sin interfaz)

In [ ]:
import mlflow
mlflow.set_tracking_uri((Path('/content/Actividad5/mlruns')).resolve().as_uri())
e=mlflow.get_experiment_by_name('Actividad5_Churn_Sintetico')
runs=mlflow.search_runs(experiment_ids=[e.experiment_id])
display(runs[['tags.mlflow.runName','metrics.f1','metrics.roc_auc','metrics.cv_f1_mean']])
assert len(runs)>=2, 'Faltan ejecuciones registradas' 

## 7. Evidencia
Toma capturas de la tabla de runs, de la carpeta `mlruns` y de la interfaz de MLflow (instrucciones en `evidencias/GUIA_MLFLOW.md`). Descarga la carpeta si necesitas abrir MLflow UI localmente. **No publiques datos personales ni credenciales.**